In [1]:
import pandas as pd
import numpy as np

# Indlæs filen
df_input = pd.read_csv('Data117/input_landbrugsdata.csv')
df_timer = pd.read_csv('Data117/Lontimer_landbrugsdata.csv')
#filter for løbende priser og foregående års priser
df_lonsum = df_input.loc[(df_input['TILGANG1'] == 'Aflønning af ansatte') & 
                         (df_input['PRISENHED'] == 'Løbende priser')].copy()
df_timerxselv = df_timer.loc[(df_timer['SOCIO'] == 'Præsterede timer for lønmodtagere (1000 timer)')].copy()
df_timer_ialt = df_timer.loc[(df_timer['SOCIO'] != 'Præsterede timer for lønmodtagere (1000 timer)')].copy()
#fjern prishenhed
df_lonsum.drop(columns=['PRISENHED'], inplace=True)
df_lonsum.drop(columns=['TILGANG1'], inplace=True)
df_timerxselv.drop(columns=['SOCIO'], inplace=True)
df_timer_ialt.drop(columns=['SOCIO'], inplace=True)

mapping = {
    '010000 Landbrug og gartneri': '010000',
    '010000 Landbrug og gartneri- (Anvendelse)': '010000',
    '100010 Slagterier': '100010',
    '100010 Slagterier- (Anvendelse)': '100010',
    '100030 Mejerier': '100030',
    '100030 Mejerier- (Anvendelse)': '100030',
    '100040x100050 Anden fødevareindustri (100040, 100050)': '100040x100050',
    '100040x100050 Anden fødevareindustri (100040, 100050)- (Anvendelse)': '100040x100050',
    'REST_TILGANG Øvrige brancher': 'REST',
    'REST_ANVENDELSE Øvrige brancher': 'REST'
}

# Erstat navnene i de to kolonner
df_timerxselv['BRANCHE'] = df_timerxselv['BRANCHE'].replace(mapping)
df_timer_ialt['BRANCHE'] = df_timer_ialt['BRANCHE'].replace(mapping)
df_lonsum['ANVENDELSE'] = df_lonsum['ANVENDELSE'].replace(mapping)

#set index
df_lonsum.set_index(['ANVENDELSE', 'TID'], inplace=True)
df_timerxselv.set_index(['BRANCHE', 'TID'], inplace=True)
df_timer_ialt.set_index(['BRANCHE', 'TID'], inplace=True)

#tjek
df_timer_ialt

INDHOLD
BRANCHE       TID          
010000        2010    98594
100010        2010    23721
100030        2010    11852
010000        1972   463150
100010        1972    49589
...                     ...
100040x100050 2018    29900
              2019    29035
              2020    29401
              2021    29055
              2022    28888

[285 rows x 1 columns]

In [2]:
df_timerxselv

INDHOLD
BRANCHE       TID          
010000        2010    50879
100010        2010    23615
100030        2010    11817
010000        1991    69831
100010        1991    37992
...                     ...
100040x100050 2018    28869
              2019    28170
              2020    28596
              2021    28177
              2022    28040

[285 rows x 1 columns]

In [3]:
# Sørg for at index-navnene er identiske, ellers kan Pandas brokke sig ved division
df_timerxselv.index.names = ['ANVENDELSE', 'TID']
df_timer_lon = df_timerxselv.reset_index()
df_timer_lon.columns = ['ANVENDELSE', 'TID', 'TIMER']
df_timer_lon.to_csv('Data117/Timer_lon_landbrugsdata.csv', index=False)
df_timer_ialt.index.names = ['ANVENDELSE', 'TID']
df_timer_output = df_timer_ialt.reset_index()
df_timer_output.columns = ['ANVENDELSE', 'TID', 'TIMER']
# 4. Gem til CSV
df_timer_output.to_csv('Data117/Timer_landbrugsdata.csv', index=False)
df_timer_output


,ANVENDELSE,TID,TIMER
0,010000,2010,98594
1,100010,2010,23721
2,100030,2010,11852
3,010000,1972,463150
4,100010,1972,49589
...,...,...,...
280,100040x100050,2018,29900
281,100040x100050,2019,29035
282,100040x100050,2020,29401
283,100040x100050,2021,29055


In [4]:
# 1. Udfør divisionen specifikt på 'INDHOLD' kolonnerne
df_timelon = df_lonsum['INDHOLD'] / (df_timerxselv['INDHOLD'] / 1000)

# 2. Gør det til et DataFrame og giv kolonnen et sigende navn
df_timelon = df_timelon.to_frame(name='TIMELOEN_KR')

# 3. FLYT INDEX TIL KOLONNER (Dette er løsningen på dit problem!)
# Når du gør dette, bliver 'ANVENDELSE' og 'TID' til rigtige kolonner i din CSV
df_timelon_output = df_timelon.reset_index()

# 4. Gem til CSV
df_timelon_output.to_csv('Data117/TimeLon_landbrugsdata.csv', index=False)

# Vis resultatet for at tjekke
df_timelon_output

,ANVENDELSE,TID,TIMELOEN_KR
0,010000,1966,4.961466
1,010000,1967,5.326186
2,010000,1968,5.644206
3,010000,1969,6.012333
4,010000,1970,5.843746
...,...,...,...
280,REST,2018,301.080970
281,REST,2019,306.638691
282,REST,2020,321.519882
283,REST,2021,321.830865


### Afgifter

In [5]:
# beregner toldsats ud fra løbende priser
df_loebende_afgift = df_input.loc[(df_input['TILGANG1'] == 'Produktskatter, netto ekskl. told og moms') & 
                         (df_input['PRISENHED'] == 'Løbende priser')]
df_loebende_moms = df_input.loc[(df_input['TILGANG1'] == 'Moms') & 
                         (df_input['PRISENHED'] == 'Løbende priser')]

# Drop kolonner
df_loebende_afgift = df_loebende_afgift.drop(columns=['PRISENHED', 'TILGANG1'])
df_loebende_moms = df_loebende_moms.drop(columns=['PRISENHED', 'TILGANG1'])

# Find fælles kolonner (alle undtagen INDHOLD)
common_cols = [c for c in df_loebende_afgift.columns if c != 'INDHOLD']

# Merge på fælles kolonner for at sikre alle brancher kommer med
merged = df_loebende_afgift.merge(
    df_loebende_moms,
    on=common_cols,
    how='outer',  # outer join så alle brancher kommer med
    suffixes=('_afgift', '_moms')
)

# Læg sammen og håndter NaN
df_samlet_afgift = (merged['INDHOLD_afgift'].fillna(0) + merged['INDHOLD_moms'].fillna(0))

# Konverter til DataFrame
df_samlet_afgift_final = merged[common_cols].copy()
df_samlet_afgift_final['afgift'] = df_samlet_afgift

# Nu har du en DataFrame med alle brancher
df_samlet_afgift_final.to_csv('Data117/landbrugsdata_afgift.csv', index=False)
df_samlet_afgift_final


,ANVENDELSE,TID,afgift
0,010000 Landbrug og gartneri- (Anvendelse),1966,-95.392
1,010000 Landbrug og gartneri- (Anvendelse),1967,-64.073
2,010000 Landbrug og gartneri- (Anvendelse),1968,-90.042
3,010000 Landbrug og gartneri- (Anvendelse),1969,-123.646
4,010000 Landbrug og gartneri- (Anvendelse),1970,-20.129
...,...,...,...
280,REST_ANVENDELSE Øvrige brancher,2018,70388.353
281,REST_ANVENDELSE Øvrige brancher,2019,67902.071
282,REST_ANVENDELSE Øvrige brancher,2020,68741.773
283,REST_ANVENDELSE Øvrige brancher,2021,74766.404
